# Reto 7: Extractor y Analizador de Logs

## Programación para Ciencia de Datos
### Instituto Politécnico Nacional
### Febrero - Julio 2026

---

## Contexto del Problema

Una empresa de tecnología necesita analizar los **logs de su servidor web** para identificar patrones de uso, errores y posibles problemas de seguridad.

Los logs contienen información valiosa pero están en formato de texto sin estructurar. Tu tarea es crear un **extractor y analizador de logs** usando expresiones regulares avanzadas.

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    ANALIZADOR DE LOGS DEL SERVIDOR                      │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   ENTRADA                                                               │
│   ───────                                                               │
│   Archivo de logs con múltiples formatos:                               │
│   • Logs de acceso HTTP                                                 │
│   • Logs de errores de aplicación                                       │
│   • Logs de autenticación                                               │
│   • Logs de base de datos                                               │
│                                                                         │
│   PROCESAMIENTO                                                         │
│   ─────────────                                                         │
│   ┌─────────────┐   ┌─────────────┐   ┌─────────────┐                   │
│   │   Parsear   │ → │  Clasificar │ → │  Analizar   │                   │
│   │   líneas    │   │   por tipo  │   │  patrones   │                   │
│   └─────────────┘   └─────────────┘   └─────────────┘                   │
│                                                                         │
│   SALIDA                                                                │
│   ──────                                                                │
│   • Estadísticas de acceso                                              │
│   • Listado de errores críticos                                         │
│   • Alertas de seguridad                                                │
│   • Datos estructurados extraídos                                       │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

## Formatos de Logs

### 1. Log de Acceso HTTP (Apache/Nginx)
```
192.168.1.100 - - [15/Mar/2024:10:23:45 -0600] "GET /api/users HTTP/1.1" 200 1234 "https://ejemplo.com" "Mozilla/5.0"
│              │   │                          │                          │    │    │                   │
└─ IP          │   └─ Fecha/Hora              └─ Método/Ruta/Protocolo   │    │    └─ Referer          └─ User-Agent
               └─ Usuario                                                │    └─ Bytes
                                                                         └─ Status
```

### 2. Log de Errores de Aplicación
```
[2024-03-15 10:25:12] ERROR app.module.submodule - DatabaseConnectionError: Connection refused to host db.server.com:5432
│                     │     │                     │                         │
└─ Timestamp          │     └─ Módulo             └─ Tipo de Error          └─ Mensaje detallado
                      └─ Nivel (ERROR/WARNING/INFO/DEBUG)
```

### 3. Log de Autenticación
```
[AUTH] 2024-03-15 10:30:00 | user=admin@empresa.com | action=LOGIN | status=SUCCESS | ip=10.0.0.5 | session=abc123xyz
[AUTH] 2024-03-15 10:31:00 | user=hacker@mail.com | action=LOGIN | status=FAILED | ip=192.168.1.50 | attempts=5
```

### 4. Log de Base de Datos
```
[DB-2024-03-15 10:35:22] QUERY executed in 0.045s: SELECT * FROM users WHERE email LIKE '%@empresa.com'
[DB-2024-03-15 10:36:00] SLOW_QUERY (2.5s): SELECT * FROM orders JOIN products ON orders.product_id = products.id
```

## Requerimientos Funcionales

### Parte 1: Parsers de Logs (40%)

Implementa parsers para cada tipo de log usando **expresiones regulares avanzadas**:

```python
def parse_http_log(linea: str) -> dict | None:
    """
    Parsea una línea de log HTTP.
    Debe usar: grupos con nombre, re.VERBOSE
    
    Retorna:
        {
            "ip": str,
            "timestamp": str,
            "method": str,
            "path": str,
            "status": int,
            "bytes": int,
            "referer": str,
            "user_agent": str
        }
    """

def parse_error_log(linea: str) -> dict | None:
    """
    Parsea una línea de log de errores.
    
    Retorna:
        {
            "timestamp": str,
            "level": str,
            "module": str,
            "error_type": str,
            "message": str
        }
    """

def parse_auth_log(linea: str) -> dict | None:
    """
    Parsea una línea de log de autenticación.
    Debe usar: lookbehind para extraer valores después de '='
    
    Retorna:
        {
            "timestamp": str,
            "user": str,
            "action": str,
            "status": str,
            "ip": str,
            "extra": dict  # session o attempts
        }
    """

def parse_db_log(linea: str) -> dict | None:
    """
    Parsea una línea de log de base de datos.
    Debe detectar: QUERY normal y SLOW_QUERY
    
    Retorna:
        {
            "timestamp": str,
            "query_type": str,  # QUERY o SLOW_QUERY
            "execution_time": float,
            "query": str
        }
    """
```

### Parte 2: Analizador de Seguridad (30%)

Implementa funciones que detecten **patrones sospechosos**:

```python
def detectar_ataques_fuerza_bruta(logs_auth: list) -> list:
    """
    Detecta intentos de fuerza bruta.
    Criterio: más de 3 intentos fallidos desde la misma IP
    
    Retorna: lista de IPs sospechosas con número de intentos
    """

def detectar_sql_injection(logs_db: list) -> list:
    """
    Detecta posibles intentos de SQL injection.
    Debe buscar: OR 1=1, UNION SELECT, --, etc.
    Usar: lookahead/lookbehind, IGNORECASE
    
    Retorna: lista de queries sospechosos
    """

def detectar_path_traversal(logs_http: list) -> list:
    """
    Detecta intentos de path traversal.
    Debe buscar: ../, ..\\ en las rutas
    
    Retorna: lista de requests sospechosos
    """

def detectar_errores_criticos(logs_error: list) -> list:
    """
    Filtra errores de nivel ERROR o CRITICAL.
    
    Retorna: lista de errores críticos ordenados por timestamp
    """
```

### Parte 3: Generador de Reportes (30%)

```python
def generar_reporte(logs: str) -> dict:
    """
    Genera un reporte completo analizando todos los logs.
    
    Retorna:
        {
            "resumen": {
                "total_lineas": int,
                "por_tipo": {"http": int, "error": int, "auth": int, "db": int}
            },
            "http": {
                "total_requests": int,
                "por_status": {"2xx": int, "3xx": int, "4xx": int, "5xx": int},
                "top_rutas": [(ruta, count), ...],
                "top_ips": [(ip, count), ...]
            },
            "errores": {
                "total": int,
                "por_nivel": {"ERROR": int, "WARNING": int, ...},
                "por_modulo": {modulo: count, ...}
            },
            "seguridad": {
                "alertas_fuerza_bruta": [...],
                "alertas_sql_injection": [...],
                "alertas_path_traversal": [...]
            },
            "rendimiento": {
                "queries_lentos": [...],
                "tiempo_promedio_queries": float
            }
        }
    """
```

## Código Base

In [1]:
import re
from typing import Dict, List, Optional, Tuple
from collections import Counter, defaultdict

In [2]:
# Patrón para log HTTP - usa re.VERBOSE para legibilidad
PATRON_HTTP = re.compile(r'''
    ^(?P<ip>\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3})    # Dirección IP
    \s+-\s+-\s+                                     # Usuario remoto e identidad (no se usan, siempre "- -")
    \[(?P<timestamp>[^\]]+)\]                       # Fecha y hora entre corchetes
    \s+"(?P<method>[A-Z]+)\s+                       # Método HTTP (GET, POST, etc.)
    (?P<path>\S+)\s+                                # Ruta solicitada
    HTTP/[\d.]+"                                     # Protocolo HTTP/version
    \s+(?P<status>\d{3})                             # Código de estado
    \s+(?P<bytes>\d+)                                # Bytes transferidos
    \s+"(?P<referer>[^"]*)"                          # Referer
    \s+"(?P<user_agent>[^"]*)"                       # User-Agent
''', re.VERBOSE)

def parse_http_log(linea: str) -> Optional[Dict]:
    """
    Parsea una línea de log HTTP.

    Ejemplo de entrada:
    192.168.1.100 - - [15/Mar/2024:10:23:45 -0600] "GET /api/users HTTP/1.1" 200 1234 "https://ejemplo.com" "Mozilla/5.0"
    """
    match = PATRON_HTTP.match(linea)
    if not match:
        return None

    datos = match.groupdict()
    return {
        "ip": datos["ip"],
        "timestamp": datos["timestamp"],
        "method": datos["method"],
        "path": datos["path"],
        "status": int(datos["status"]),
        "bytes": int(datos["bytes"]),
        "referer": datos["referer"],
        "user_agent": datos["user_agent"],
    }

In [3]:
# Patrón para log de errores
PATRON_ERROR = re.compile(r'''
    ^\[(?P<timestamp>\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2})\]   # Timestamp entre corchetes
    \s+(?P<level>ERROR|WARNING|INFO|DEBUG|CRITICAL)              # Nivel del log
    \s+(?P<module>[\w.]+)                                        # Módulo (puede tener puntos)
    \s+-\s+
    (?P<error_type>\w+)                                          # Tipo de error
    :\s*
    (?P<message>.+)$                                             # Mensaje detallado
''', re.VERBOSE)

def parse_error_log(linea: str) -> Optional[Dict]:
    """
    Parsea una línea de log de errores.

    Ejemplo de entrada:
    [2024-03-15 10:25:12] ERROR app.module.submodule - DatabaseConnectionError: Connection refused
    """
    match = PATRON_ERROR.match(linea)
    if not match:
        return None

    datos = match.groupdict()
    return {
        "timestamp": datos["timestamp"],
        "level": datos["level"],
        "module": datos["module"],
        "error_type": datos["error_type"],
        "message": datos["message"],
    }

In [4]:
# Patrón para log de autenticación
PATRON_AUTH = re.compile(r'''
    ^\[AUTH\]\s+
    (?P<timestamp>\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2})\s*\|\s*
    user=(?P<user>[^\s|]+)\s*\|\s*
    action=(?P<action>[^\s|]+)\s*\|\s*
    status=(?P<status>[^\s|]+)\s*\|\s*
    ip=(?P<ip>[^\s|]+)
    (?:\s*\|\s*(?P<extra_key>session|attempts)=(?P<extra_value>[^\s|]+))?   # campo extra opcional
''', re.VERBOSE)

def parse_auth_log(linea: str) -> Optional[Dict]:
    """
    Parsea una línea de log de autenticación.
    Usa lookbehind para extraer valores después de '='.

    Ejemplo de entrada:
    [AUTH] 2024-03-15 10:30:00 | user=admin@empresa.com | action=LOGIN | status=SUCCESS | ip=10.0.0.5
    """
    match = PATRON_AUTH.match(linea)
    if not match:
        return None

    datos = match.groupdict()

    # Lookbehind para extraer los valores que vienen después de cada '='
    # (?<=clave=) afirma que justo antes de la posición actual existe "clave="
    # sin incluir ese texto en el resultado capturado.
    valores_extra = {}
    for clave in ("session", "attempts"):
        patron_lookbehind = rf'(?<={clave}=)[^\s|]+'
        encontrado = re.search(patron_lookbehind, linea)
        if encontrado:
            valor = encontrado.group()
            valores_extra[clave] = int(valor) if clave == "attempts" else valor

    return {
        "timestamp": datos["timestamp"],
        "user": datos["user"],
        "action": datos["action"],
        "status": datos["status"],
        "ip": datos["ip"],
        "extra": valores_extra,
    }

In [5]:
# Patrón para log de base de datos
PATRON_DB = re.compile(r'''
    ^\[DB-(?P<timestamp>\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2})\]\s+
    (?:
        QUERY\s+executed\s+in\s+(?P<tiempo_query>[\d.]+)s          # caso QUERY normal
        |
        SLOW_QUERY\s+\((?P<tiempo_slow>[\d.]+)s\)                  # caso SLOW_QUERY
    )
    :\s*(?P<query>.+)$                                              # Texto del query
''', re.VERBOSE)

def parse_db_log(linea: str) -> Optional[Dict]:
    """
    Parsea una línea de log de base de datos.
    Detecta QUERY y SLOW_QUERY.

    Ejemplos de entrada:
    [DB-2024-03-15 10:35:22] QUERY executed in 0.045s: SELECT * FROM users
    [DB-2024-03-15 10:36:00] SLOW_QUERY (2.5s): SELECT * FROM orders JOIN products
    """
    match = PATRON_DB.match(linea)
    if not match:
        return None

    datos = match.groupdict()
    es_lenta = datos["tiempo_slow"] is not None
    tiempo = datos["tiempo_slow"] if es_lenta else datos["tiempo_query"]

    return {
        "timestamp": datos["timestamp"],
        "query_type": "SLOW_QUERY" if es_lenta else "QUERY",
        "execution_time": float(tiempo),
        "query": datos["query"],
    }

In [6]:
def detectar_ataques_fuerza_bruta(logs_auth: List[Dict]) -> List[Dict]:
    """
    Detecta IPs con múltiples intentos fallidos de login.
    Criterio: más de 3 intentos FAILED desde la misma IP.
    """
    intentos_fallidos_por_ip = Counter()

    for log in logs_auth:
        if log.get("action") == "LOGIN" and log.get("status") == "FAILED":
            intentos_fallidos_por_ip[log["ip"]] += 1

    alertas = [
        {"ip": ip, "intentos": intentos}
        for ip, intentos in intentos_fallidos_por_ip.items()
        if intentos > 3
    ]
    return alertas

In [7]:
# Patrones de SQL Injection a detectar
PATRONES_SQL_INJECTION = [
    r"(?i)\bOR\b\s+['\"]?\d+['\"]?\s*=\s*['\"]?\d+",  # OR 1=1
    r"(?i)\bUNION\b.*\bSELECT\b",                       # UNION SELECT
    r"(?i)\bDROP\b\s+\bTABLE\b",                        # DROP TABLE
    r"(?i)\bDELETE\b\s+\bFROM\b.*\bWHERE\b\s+1\s*=\s*1", # DELETE WHERE 1=1
]

# Patrón con lookbehind: detecta el comentario SQL "--" únicamente cuando
# está precedido por una comilla (simple o doble). Ese es el caso típico de
# un injection que intenta "cerrar" la cadena del query y comentar el resto,
# y evita falsos positivos con "--" en otros contextos.
PATRON_COMENTARIO_SQL = re.compile(r"(?<=['\"])\s*--")

def detectar_sql_injection(logs_db: List[Dict]) -> List[Dict]:
    """
    Detecta posibles intentos de SQL injection en las queries.
    """
    alertas = []

    for log in logs_db:
        query = log.get("query", "")
        sospechoso = (
            any(re.search(patron, query) for patron in PATRONES_SQL_INJECTION)
            or PATRON_COMENTARIO_SQL.search(query)
        )

        if sospechoso:
            alertas.append({
                "timestamp": log.get("timestamp"),
                "query": query,
            })

    return alertas

In [8]:
def detectar_path_traversal(logs_http: List[Dict]) -> List[Dict]:
    """
    Detecta intentos de path traversal en las rutas HTTP.
    Busca patrones como: ../, ..\\, %2e%2e%2f
    """
    patron_traversal = re.compile(r'''
        \.\.[/\\]                 # secuencia ../ o ..(barra invertida)
        |
        %2e%2e%2f                 # ../ codificado en URL
    ''', re.VERBOSE | re.IGNORECASE)

    alertas = []
    for log in logs_http:
        path = log.get("path", "")
        if patron_traversal.search(path):
            alertas.append({
                "ip": log.get("ip"),
                "path": path,
            })

    return alertas

In [9]:
def detectar_errores_criticos(logs_error: List[Dict]) -> List[Dict]:
    """
    Filtra y retorna errores de nivel ERROR o CRITICAL.
    """
    criticos = [
        log for log in logs_error
        if log.get("level") in ("ERROR", "CRITICAL")
    ]
    return sorted(criticos, key=lambda log: log["timestamp"])

In [10]:
def clasificar_linea(linea: str) -> str:
    """
    Clasifica una línea de log por su tipo.
    Retorna: 'http', 'error', 'auth', 'db', o 'desconocido'
    """
    if linea.startswith("[AUTH]"):
        return "auth"
    if linea.startswith("[DB-"):
        return "db"
    if re.match(r'^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\s+-\s+-\s+\[', linea):
        return "http"
    if re.match(r'^\[\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2}\]', linea):
        return "error"
    return "desconocido"


def _clasificar_status(status: int) -> str:
    """Convierte un código de estado HTTP en su categoría (2xx, 3xx, 4xx, 5xx)."""
    return f"{status // 100}xx"


def generar_reporte(logs: str) -> Dict:
    """
    Genera un reporte completo analizando todos los logs.
    """
    lineas = [linea for linea in logs.strip().split("\n") if linea.strip()]

    logs_http, logs_error, logs_auth, logs_db = [], [], [], []
    por_tipo = Counter()

    parsers = {
        "http": (parse_http_log, logs_http),
        "error": (parse_error_log, logs_error),
        "auth": (parse_auth_log, logs_auth),
        "db": (parse_db_log, logs_db),
    }

    for linea in lineas:
        tipo = clasificar_linea(linea)
        por_tipo[tipo] += 1

        if tipo in parsers:
            parser, contenedor = parsers[tipo]
            resultado = parser(linea)
            if resultado is not None:
                contenedor.append(resultado)

    # --- Estadísticas HTTP ---
    por_status = Counter()
    for log in logs_http:
        por_status[_clasificar_status(log["status"])] += 1
    for categoria in ("2xx", "3xx", "4xx", "5xx"):
        por_status.setdefault(categoria, 0)

    contador_rutas = Counter(log["path"] for log in logs_http)
    contador_ips = Counter(log["ip"] for log in logs_http)

    http_stats = {
        "total_requests": len(logs_http),
        "por_status": dict(por_status),
        "top_rutas": contador_rutas.most_common(5),
        "top_ips": contador_ips.most_common(5),
    }

    # --- Estadísticas de errores ---
    por_nivel = Counter(log["level"] for log in logs_error)
    por_modulo = Counter(log["module"] for log in logs_error)

    errores_stats = {
        "total": len(logs_error),
        "por_nivel": dict(por_nivel),
        "por_modulo": dict(por_modulo),
    }

    # --- Seguridad ---
    seguridad_stats = {
        "alertas_fuerza_bruta": detectar_ataques_fuerza_bruta(logs_auth),
        "alertas_sql_injection": detectar_sql_injection(logs_db),
        "alertas_path_traversal": detectar_path_traversal(logs_http),
    }

    # --- Rendimiento ---
    queries_lentos = [log for log in logs_db if log["query_type"] == "SLOW_QUERY"]
    tiempos = [log["execution_time"] for log in logs_db]
    tiempo_promedio = sum(tiempos) / len(tiempos) if tiempos else 0.0

    rendimiento_stats = {
        "queries_lentos": queries_lentos,
        "tiempo_promedio_queries": tiempo_promedio,
    }

    return {
        "resumen": {
            "total_lineas": len(lineas),
            "por_tipo": {
                "http": por_tipo.get("http", 0),
                "error": por_tipo.get("error", 0),
                "auth": por_tipo.get("auth", 0),
                "db": por_tipo.get("db", 0),
            },
        },
        "http": http_stats,
        "errores": errores_stats,
        "seguridad": seguridad_stats,
        "rendimiento": rendimiento_stats,
    }

## Datos de Prueba

In [11]:
LOGS_PRUEBA = """
192.168.1.100 - - [15/Mar/2024:10:23:45 -0600] "GET /api/users HTTP/1.1" 200 1234 "https://ejemplo.com" "Mozilla/5.0 (Windows NT 10.0)"
192.168.1.101 - - [15/Mar/2024:10:23:46 -0600] "POST /api/login HTTP/1.1" 200 89 "-" "curl/7.68.0"
192.168.1.102 - - [15/Mar/2024:10:23:47 -0600] "GET /admin/../../../etc/passwd HTTP/1.1" 403 0 "-" "sqlmap/1.0"
[2024-03-15 10:24:00] INFO app.startup - Application started successfully on port 8080
[2024-03-15 10:25:12] ERROR app.database - DatabaseConnectionError: Connection refused to host db.server.com:5432
[2024-03-15 10:25:15] WARNING app.cache - CacheWarning: Redis connection timeout, using fallback
[2024-03-15 10:26:00] ERROR app.auth - AuthenticationError: Invalid token for user admin@empresa.com
[AUTH] 2024-03-15 10:30:00 | user=admin@empresa.com | action=LOGIN | status=SUCCESS | ip=10.0.0.5 | session=abc123xyz
[AUTH] 2024-03-15 10:31:00 | user=hacker@mail.com | action=LOGIN | status=FAILED | ip=192.168.1.50 | attempts=1
[AUTH] 2024-03-15 10:31:30 | user=hacker@mail.com | action=LOGIN | status=FAILED | ip=192.168.1.50 | attempts=2
[AUTH] 2024-03-15 10:32:00 | user=hacker@mail.com | action=LOGIN | status=FAILED | ip=192.168.1.50 | attempts=3
[AUTH] 2024-03-15 10:32:30 | user=hacker@mail.com | action=LOGIN | status=FAILED | ip=192.168.1.50 | attempts=4
[AUTH] 2024-03-15 10:33:00 | user=otro@empresa.com | action=LOGOUT | status=SUCCESS | ip=10.0.0.10 | session=def456uvw
[DB-2024-03-15 10:35:22] QUERY executed in 0.045s: SELECT * FROM users WHERE email = 'admin@empresa.com'
[DB-2024-03-15 10:35:25] QUERY executed in 0.012s: SELECT id, name FROM products WHERE active = 1
[DB-2024-03-15 10:36:00] SLOW_QUERY (2.5s): SELECT * FROM orders o JOIN products p ON o.product_id = p.id JOIN users u ON o.user_id = u.id
[DB-2024-03-15 10:37:00] QUERY executed in 0.001s: SELECT * FROM users WHERE username = 'admin' OR 1=1--'
[DB-2024-03-15 10:38:00] QUERY executed in 0.002s: SELECT * FROM users UNION SELECT * FROM passwords
192.168.1.200 - - [15/Mar/2024:10:40:00 -0600] "GET /products?id=1 HTTP/1.1" 200 5678 "https://tienda.com" "Mozilla/5.0"
192.168.1.200 - - [15/Mar/2024:10:40:05 -0600] "GET /products?id=2 HTTP/1.1" 200 4321 "https://tienda.com" "Mozilla/5.0"
192.168.1.201 - - [15/Mar/2024:10:41:00 -0600] "GET /api/users HTTP/1.1" 401 123 "-" "PostmanRuntime/7.26.8"
192.168.1.201 - - [15/Mar/2024:10:41:05 -0600] "GET /api/users HTTP/1.1" 500 0 "-" "PostmanRuntime/7.26.8"
[2024-03-15 10:42:00] ERROR app.api - NullPointerException: Cannot read property 'id' of undefined
[DB-2024-03-15 10:45:00] SLOW_QUERY (5.2s): SELECT COUNT(*) FROM logs WHERE date > '2024-01-01'
""".strip()

## Funciones de Visualización

In [12]:
def mostrar_reporte(reporte: Dict) -> None:
    """Muestra el reporte de forma legible."""
    print("=" * 70)
    print("                    REPORTE DE ANÁLISIS DE LOGS")
    print("=" * 70)
    
    # Resumen
    print("\n📊 RESUMEN GENERAL")
    print("-" * 40)
    print(f"Total de líneas procesadas: {reporte['resumen']['total_lineas']}")
    print("Por tipo:")
    for tipo, count in reporte['resumen']['por_tipo'].items():
        print(f"  • {tipo.upper()}: {count}")
    
    # HTTP
    if 'http' in reporte:
        print("\n🌐 LOGS HTTP")
        print("-" * 40)
        print(f"Total requests: {reporte['http']['total_requests']}")
        print("Por código de estado:")
        for status, count in reporte['http']['por_status'].items():
            print(f"  • {status}: {count}")
        print("Top 5 rutas más solicitadas:")
        for ruta, count in reporte['http'].get('top_rutas', [])[:5]:
            print(f"  • {ruta}: {count} requests")
    
    # Errores
    if 'errores' in reporte:
        print("\n❌ ERRORES")
        print("-" * 40)
        print(f"Total errores: {reporte['errores']['total']}")
        print("Por nivel:")
        for nivel, count in reporte['errores']['por_nivel'].items():
            print(f"  • {nivel}: {count}")
    
    # Seguridad
    if 'seguridad' in reporte:
        print("\n🔒 ALERTAS DE SEGURIDAD")
        print("-" * 40)
        
        fb = reporte['seguridad'].get('alertas_fuerza_bruta', [])
        if fb:
            print(f"⚠️  Posibles ataques de fuerza bruta: {len(fb)}")
            for alerta in fb:
                print(f"     IP: {alerta['ip']} - {alerta['intentos']} intentos fallidos")
        
        sql = reporte['seguridad'].get('alertas_sql_injection', [])
        if sql:
            print(f"⚠️  Posibles SQL Injection: {len(sql)}")
            for alerta in sql[:3]:  # Mostrar solo las primeras 3
                print(f"     Query: {alerta['query'][:60]}...")
        
        pt = reporte['seguridad'].get('alertas_path_traversal', [])
        if pt:
            print(f"⚠️  Posibles Path Traversal: {len(pt)}")
            for alerta in pt[:3]:
                print(f"     Ruta: {alerta['path']}")
    
    # Rendimiento
    if 'rendimiento' in reporte:
        print("\n⏱️  RENDIMIENTO")
        print("-" * 40)
        print(f"Queries lentos detectados: {len(reporte['rendimiento'].get('queries_lentos', []))}")
        if 'tiempo_promedio_queries' in reporte['rendimiento']:
            print(f"Tiempo promedio de queries: {reporte['rendimiento']['tiempo_promedio_queries']:.3f}s")
    
    print("\n" + "=" * 70)

## Prueba tu Implementación

In [13]:
# Prueba de parsers individuales
print("PRUEBA DE PARSERS")
print("=" * 50)

# HTTP
linea_http = '192.168.1.100 - - [15/Mar/2024:10:23:45 -0600] "GET /api/users HTTP/1.1" 200 1234 "https://ejemplo.com" "Mozilla/5.0"'
print("\n-- Parser HTTP --")
print(f"Entrada: {linea_http[:60]}...")
print(f"Resultado: {parse_http_log(linea_http)}")

# Error
linea_error = "[2024-03-15 10:25:12] ERROR app.database - DatabaseConnectionError: Connection refused"
print("\n-- Parser Error --")
print(f"Entrada: {linea_error}")
print(f"Resultado: {parse_error_log(linea_error)}")

# Auth
linea_auth = "[AUTH] 2024-03-15 10:30:00 | user=admin@empresa.com | action=LOGIN | status=SUCCESS | ip=10.0.0.5 | session=abc123xyz"
print("\n-- Parser Auth --")
print(f"Entrada: {linea_auth}")
print(f"Resultado: {parse_auth_log(linea_auth)}")

# DB
linea_db = "[DB-2024-03-15 10:35:22] QUERY executed in 0.045s: SELECT * FROM users"
print("\n-- Parser DB --")
print(f"Entrada: {linea_db}")
print(f"Resultado: {parse_db_log(linea_db)}")

PRUEBA DE PARSERS

-- Parser HTTP --
Entrada: 192.168.1.100 - - [15/Mar/2024:10:23:45 -0600] "GET /api/use...
Resultado: {'ip': '192.168.1.100', 'timestamp': '15/Mar/2024:10:23:45 -0600', 'method': 'GET', 'path': '/api/users', 'status': 200, 'bytes': 1234, 'referer': 'https://ejemplo.com', 'user_agent': 'Mozilla/5.0'}

-- Parser Error --
Entrada: [2024-03-15 10:25:12] ERROR app.database - DatabaseConnectionError: Connection refused
Resultado: {'timestamp': '2024-03-15 10:25:12', 'level': 'ERROR', 'module': 'app.database', 'error_type': 'DatabaseConnectionError', 'message': 'Connection refused'}

-- Parser Auth --
Entrada: [AUTH] 2024-03-15 10:30:00 | user=admin@empresa.com | action=LOGIN | status=SUCCESS | ip=10.0.0.5 | session=abc123xyz
Resultado: {'timestamp': '2024-03-15 10:30:00', 'user': 'admin@empresa.com', 'action': 'LOGIN', 'status': 'SUCCESS', 'ip': '10.0.0.5', 'extra': {'session': 'abc123xyz'}}

-- Parser DB --
Entrada: [DB-2024-03-15 10:35:22] QUERY executed in 0.045s: SELEC

In [14]:
# Prueba del reporte completo
print("\nGENERANDO REPORTE COMPLETO...\n")
reporte = generar_reporte(LOGS_PRUEBA)
mostrar_reporte(reporte)


GENERANDO REPORTE COMPLETO...

                    REPORTE DE ANÁLISIS DE LOGS

📊 RESUMEN GENERAL
----------------------------------------
Total de líneas procesadas: 24
Por tipo:
  • HTTP: 7
  • ERROR: 5
  • AUTH: 6
  • DB: 6

🌐 LOGS HTTP
----------------------------------------
Total requests: 7
Por código de estado:
  • 2xx: 4
  • 4xx: 2
  • 5xx: 1
  • 3xx: 0
Top 5 rutas más solicitadas:
  • /api/users: 3 requests
  • /api/login: 1 requests
  • /admin/../../../etc/passwd: 1 requests
  • /products?id=1: 1 requests
  • /products?id=2: 1 requests

❌ ERRORES
----------------------------------------
Total errores: 4
Por nivel:
  • ERROR: 3
  • WARNING: 1

🔒 ALERTAS DE SEGURIDAD
----------------------------------------
⚠️  Posibles ataques de fuerza bruta: 1
     IP: 192.168.1.50 - 4 intentos fallidos
⚠️  Posibles SQL Injection: 2
     Query: SELECT * FROM users WHERE username = 'admin' OR 1=1--'...
     Query: SELECT * FROM users UNION SELECT * FROM passwords...
⚠️  Posibles Path Traver

## Salida Esperada

```
======================================================================
                    REPORTE DE ANÁLISIS DE LOGS
======================================================================

📊 RESUMEN GENERAL
----------------------------------------
Total de líneas procesadas: 23
Por tipo:
  • HTTP: 7
  • ERROR: 5
  • AUTH: 6
  • DB: 5

🌐 LOGS HTTP
----------------------------------------
Total requests: 7
Por código de estado:
  • 2xx: 4
  • 4xx: 2
  • 5xx: 1
Top 5 rutas más solicitadas:
  • /api/users: 3 requests
  • /products: 2 requests

❌ ERRORES
----------------------------------------
Total errores: 5
Por nivel:
  • ERROR: 3
  • WARNING: 1
  • INFO: 1

🔒 ALERTAS DE SEGURIDAD
----------------------------------------
⚠️  Posibles ataques de fuerza bruta: 1
     IP: 192.168.1.50 - 4 intentos fallidos
⚠️  Posibles SQL Injection: 2
     Query: SELECT * FROM users WHERE username = 'admin' OR 1=1--...
     Query: SELECT * FROM users UNION SELECT * FROM passwords...
⚠️  Posibles Path Traversal: 1
     Ruta: /admin/../../../etc/passwd

⏱️  RENDIMIENTO
----------------------------------------
Queries lentos detectados: 2
Tiempo promedio de queries: 1.552s

======================================================================
```

## Bonus: Funcionalidades Extra (Opcional)

### 1. Exportar a JSON

In [15]:
import json

def exportar_reporte_json(reporte: Dict, archivo: str) -> None:
    """Exporta el reporte a un archivo JSON."""
    with open(archivo, "w", encoding="utf-8") as f:
        json.dump(reporte, f, indent=4, ensure_ascii=False)

### 2. Análisis temporal

In [16]:
def analisis_temporal(logs_http: List[Dict]) -> Dict:
    """
    Analiza distribución de requests por hora.

    Retorna:
        {"hora": count, ...}
    """
    patron_hora = re.compile(r':(?P<hora>\d{2}):\d{2}:\d{2}\s')
    conteo_por_hora = Counter()

    for log in logs_http:
        match = patron_hora.search(log["timestamp"])
        if match:
            conteo_por_hora[match.group("hora")] += 1

    return dict(sorted(conteo_por_hora.items()))

### 3. Detección de bots

In [17]:
# User-Agents conocidos de bots/herramientas automatizadas
PATRON_BOTS = re.compile(r'''
    (?i)
    curl|wget|python-requests|scrapy|sqlmap|
    bot|spider|crawler|postmanruntime
''', re.VERBOSE)

def detectar_bots(logs_http: List[Dict]) -> List[Dict]:
    """
    Detecta requests que parecen venir de bots.
    Busca User-Agents conocidos de bots: curl, wget, python-requests, scrapy, etc.
    """
    return [
        log for log in logs_http
        if PATRON_BOTS.search(log.get("user_agent", ""))
    ]

### Prueba de las funcionalidades Bonus

In [18]:
# Prueba de las funcionalidades bonus
print("PRUEBA DE BONUS")
print("=" * 50)

# Reconstruimos la lista de logs HTTP ya parseados a partir de LOGS_PRUEBA
_lineas = [l for l in LOGS_PRUEBA.strip().split("\n") if l.strip()]
_logs_http = [parse_http_log(l) for l in _lineas if clasificar_linea(l) == "http"]
_logs_http = [l for l in _logs_http if l is not None]

# 1. Exportar a JSON
exportar_reporte_json(reporte, "reporte_logs.json")
print("\n-- Exportar JSON --")
print("Reporte exportado correctamente a 'reporte_logs.json'")

# 2. Análisis temporal
print("\n-- Análisis temporal (requests por hora) --")
print(analisis_temporal(_logs_http))

# 3. Detección de bots
print("\n-- Detección de bots --")
bots_detectados = detectar_bots(_logs_http)
for bot in bots_detectados:
    print(f"  • IP {bot['ip']} -> User-Agent: {bot['user_agent']}")

PRUEBA DE BONUS

-- Exportar JSON --
Reporte exportado correctamente a 'reporte_logs.json'

-- Análisis temporal (requests por hora) --
{'10': 7}

-- Detección de bots --
  • IP 192.168.1.101 -> User-Agent: curl/7.68.0
  • IP 192.168.1.102 -> User-Agent: sqlmap/1.0
  • IP 192.168.1.201 -> User-Agent: PostmanRuntime/7.26.8
  • IP 192.168.1.201 -> User-Agent: PostmanRuntime/7.26.8


---

## Criterios de Evaluación

| Criterio | Puntos |
|----------|--------|
| **Parte 1: Parsers de Logs** | 40 |
| - `parse_http_log()` con grupos nombrados y VERBOSE | 10 |
| - `parse_error_log()` funciona correctamente | 10 |
| - `parse_auth_log()` usa lookbehind | 10 |
| - `parse_db_log()` detecta QUERY y SLOW_QUERY | 10 |
| **Parte 2: Analizador de Seguridad** | 30 |
| - Detecta ataques de fuerza bruta | 8 |
| - Detecta SQL injection con patrones avanzados | 8 |
| - Detecta path traversal | 7 |
| - Filtra errores críticos | 7 |
| **Parte 3: Generador de Reportes** | 30 |
| - Clasifica líneas correctamente | 10 |
| - Genera estadísticas HTTP (status, rutas, IPs) | 10 |
| - Integra análisis de seguridad y rendimiento | 10 |
| **Bonus** | +15 |
| **Total** | 100 (+15) |

---

## Entregables

1. Notebook con todas las funciones implementadas
2. Todas las celdas ejecutadas mostrando resultados
3. Patrones regex documentados con `re.VERBOSE`
4. Al menos 2 patrones deben usar lookahead o lookbehind

---

## Consejos

- Usa [regex101.com](https://regex101.com/) con la opción "Python" para probar
- Desarrolla y prueba cada parser por separado antes de integrar
- Usa `re.VERBOSE` para documentar patrones complejos
- Compila patrones que uses múltiples veces
- Maneja casos donde el parsing falle (retorna `None`)